## Set up a corpura

In [8]:
corpus = [
    ["I absolutely love this product and would buy it again", 1],
    ["The experience was frustrating and disappointing", 0],
    ["This was an outstanding purchase with excellent quality", 1],
    ["The service was poor and the item arrived damaged", 0],
    ["Very satisfied with the fast delivery and great design", 1],
    ["I regret buying this because it is low quality", 0],
    ["My order was perfect and I am very pleased", 1],
    ["This item is awful and not worth the price", 0],
]

words = list(dict.fromkeys(
    word.lower()
    for sentence, _ in corpus
    for word in sentence.split()
))
embedding_dim = len(words)

D = {
    word: [int(i == word_index) for i in range(embedding_dim)]
    for word_index, word in enumerate(words)
}

# D

In [9]:
# Word to ID
word_to_id = {word: word_id for word_id, word in enumerate(words)}
# word_to_id

## Initialize the Weight matrix

In [10]:
import numpy as np

# Shape: embedding dimension × vocabulary size
yweights = np.random.uniform(
    low=-0.5,
    high=0.5,
    size=(embedding_dim, len(words))
)

# Creating Training Pairs

In [11]:
window_size = 2
training_pairs = []

for sentence, _ in corpus:
    tokens = [word.lower() for word in sentence.split()]
    for center_index, center_word in enumerate(tokens):
        start = max(0, center_index - window_size)
        end = min(len(tokens), center_index + window_size + 1)
        for context_index in range(start, end):
            if context_index != center_index:
                context_word = tokens[context_index]
                training_pairs.append((word_to_id[center_word], word_to_id[context_word]))

training_pairs[:10]

[(0, 1),
 (0, 2),
 (1, 0),
 (1, 2),
 (1, 3),
 (2, 0),
 (2, 1),
 (2, 3),
 (2, 4),
 (3, 1)]

## Step 5: Train Skip-gram

In [12]:
# Start input embeddings as one-hot vectors; training will update them.
# E = np.random.uniform(
#     low=-0.5,
#     high=0.5,
#     size=(len(words), embedding_dim),
# )
E = np.array([D[word] for word in words], dtype=float)

learning_rate = 0.01
num_epochs = 10
epoch_losses = []

for epoch in range(num_epochs):
    np.random.shuffle(training_pairs)
    total_loss = 0.0

    for center_id, context_id in training_pairs:
        # Forward pass
        h = E[center_id].copy()
        scores = h @ yweights
        exp_scores = np.exp(scores - np.max(scores))
        probabilities = exp_scores / np.sum(exp_scores)

        # Cross-entropy loss and gradients
        total_loss -= np.log(probabilities[context_id] + 1e-12)
        error = probabilities.copy()
        error[context_id] -= 1
        gradient_yweights = np.outer(h, error)
        gradient_h = yweights @ error

        # SGD parameter update
        yweights -= learning_rate * gradient_yweights
        E[center_id] -= learning_rate * gradient_h

    average_loss = total_loss / len(training_pairs)
    epoch_losses.append(average_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, loss: {average_loss:.4f}")

Epoch 1/10, loss: 3.8639
Epoch 2/10, loss: 3.8237
Epoch 3/10, loss: 3.7846
Epoch 4/10, loss: 3.7462
Epoch 5/10, loss: 3.7084
Epoch 6/10, loss: 3.6711
Epoch 7/10, loss: 3.6340
Epoch 8/10, loss: 3.5971
Epoch 9/10, loss: 3.5603
Epoch 10/10, loss: 3.5233
